In [3]:
!git clone https://github.com/DataTalksClub/machine-learning-zoomcamp.git

Cloning into 'machine-learning-zoomcamp'...
remote: Enumerating objects: 18055, done.
remote: Counting objects: 100% (956/956), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 18055 (delta 921), reused 903 (delta 903), pack-reused 17099 (from 4)
Receiving objects: 100% (18055/18055), 996.13 MiB | 36.96 MiB/s, done.
Resolving deltas: 100% (11282/11282), done.


In [4]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment


In [5]:
!ls -lah

total 100K
drwxr-xr-x  2 root root 4.0K Nov  9 13:08 .
drwxr-xr-x 11 root root 4.0K Nov  9 13:08 ..
-rw-r--r--  1 root root  598 Nov  9 13:08 Dockerfile
-rw-r--r--  1 root root  116 Nov  9 13:08 feature_defaults.json
-rw-r--r--  1 root root 6.0K Nov  9 13:08 homework.md
-rw-r--r--  1 root root  563 Nov  9 13:08 homework.yaml
-rw-r--r--  1 root root  872 Nov  9 13:08 model_metadata.json
-rw-r--r--  1 root root 1.5K Nov  9 13:08 model.py
-rw-r--r--  1 root root 1.9K Nov  9 13:08 pipeline.bin
-rw-r--r--  1 root root 1.4K Nov  9 13:08 predict.py
-rw-r--r--  1 root root  326 Nov  9 13:08 pyproject.toml
-rw-r--r--  1 root root    8 Nov  9 13:08 .python-version
-rw-r--r--  1 root root  828 Nov  9 13:08 q6_test.py
-rw-r--r--  1 root root 1.1K Nov  9 13:08 README.md
-rw-r--r--  1 root root  777 Nov  9 13:08 smoke_test.py
-rw-r--r--  1 root root 3.3K Nov  9 13:08 train.py
-rw-r--r--  1 root root  32K Nov  9 13:08 uv.lock


In [ ]:
!pip install -q uv

In [12]:
!pip install -q uv
!uv --version
!uv python install 3.11.15

uv 0.13.7 (x86_64-unknown-linux-gnu)
Python 3.11.15 is already installed


In [13]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


In [14]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [7]:
!uv --version

uv 0.13.7 (x86_64-unknown-linux-gnu)


In [8]:
!uv python install 3.11.15

Installed Python 3.11.15 in 1.10s
 + cpython-3.11.15-linux-x86_64-gnu (python3.11)


In [9]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


In [10]:
!uv run python smoke_test.py

Using CPython 3.11.15
Creating virtual environment at: .venv
Installed 18 packages in 74ms
{'conversion_probability': 0.769799, 'conversion': True}


In [15]:
%%writefile q3_test.py

import pickle

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

with open("pipeline.bin", "rb") as f:
    model = pickle.load(f)

probability = model.predict_proba([lead])[0, 1]

print("Conversion probability:", round(probability, 3))

Writing q3_test.py


In [16]:
!uv run python q3_test.py

Conversion probability: 0.533


In [17]:
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696 &

INFO:     Started server process [5735]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5735]


In [19]:
!curl -s http://localhost:9696/health

In [21]:
import subprocess

server = subprocess.Popen(
    [
        "uv", "run", "uvicorn",
        "predict:app",
        "--host", "0.0.0.0",
        "--port", "9696"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("API process started:", server.pid)

API process started: 9359


In [22]:
import requests
import time

for i in range(10):
    try:
        response = requests.get("http://127.0.0.1:9696/health")
        print(response.status_code)
        print(response.json())
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    print("API did not start. We need to inspect the server logs.")

200
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [23]:
import requests

lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

response = requests.post(
    "http://127.0.0.1:9696/predict",
    json=lead
)

print(response.status_code)
print(response.json())

200
{'conversion_probability': 0.769799, 'conversion': True}


In [24]:
!cat Dockerfile

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]


In [25]:
!docker --version

/bin/bash: line 1: docker: command not found
